# Example — servicing dashboard

Collections performance by servicer, pulled from Snowflake into pandas.

**This is a worked example.** It is kept runnable against real tables
rather than stubbed out, so you can run it end to end and then copy the
parts you need into your own folder. Everything in `examples/` is prefixed
`example_`; nothing else in the workspace depends on it, so it is safe to
edit or delete.

**Before you run anything:** pick the `Python (local-workspace)` kernel in
the top-right. If it isn't offered, run `mise run kernel` — see
[../README.md](../README.md).

**First query of the day** opens a browser tab for single sign-on, and macOS
may ask for your *Mac* login password to read the Keychain (click **Always
Allow**). After that it's silent.

Everything here is **read-only** — see the last section.

## 1 — Setup

`session_info()` confirms your login works and shows what you're pointed at.
`warehouse` should read `NPL_ANALYST_WH`.

In [ ]:
import pandas as pd
import plotly.express as px

import datalake

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 250)
# Show up to 5 decimal places, trimming trailing zeros. 2dp hides real
# differences in rates and per-unit figures. This is display only — the
# underlying values keep full precision.
pd.set_option(
    "display.float_format",
    lambda v: f"{v:,.5f}".rstrip("0").rstrip("."),
)

DB = "PROD"

datalake.session_info()

## 2 — Load the data

The query lives in [example_servicing-dashboard.sql](example_servicing-dashboard.sql) next to
this notebook. `datalake` searches this folder first, so the bare filename
resolves.

That file is **plain, valid Snowflake SQL with no templating**, which means
it runs three ways without editing:

| Where | How |
|---|---|
| This notebook | `datalake.q("example_servicing-dashboard.sql")` |
| Terminal | `mise run sql -- examples/example_servicing-dashboard.sql` |
| Snowflake VS Code extension | open the file and hit Run |

The reporting window is a **Snowflake session variable** — `set months = 12;`
on the first line of the file — rather than a Python placeholder. That is the
whole reason the extension can run it: no string substitution is involved, so
Snowflake sees valid SQL either way. To change the window, edit that line.

[example_collections_by_servicer.sql](example_collections_by_servicer.sql) in
this same folder is the compact version of the same question, and uses the
same style. **Every `.sql` file in this workspace does** — verified against
the warehouse on 2026-09-09 by sending each one verbatim and compiling it, so
any of them can be opened and run in the extension.

`datalake` also supports Python `{db}` placeholders for object names, which
are more flexible from a notebook but *cannot* be run by a SQL editor. Keep
those **inline in a notebook**, as in section 8 below — not in a `.sql` file.

In [ ]:
# Read the SQL without running it, to see exactly what will execute
print(datalake.read_sql("example_servicing-dashboard.sql"))

In [ ]:
df = datalake.q("example_servicing-dashboard.sql")

print(f"{len(df)} rows, {df['MONTH'].min():%Y-%m} to {df['MONTH'].max():%Y-%m}")
df.head()

Column names come back **UPPER-CASE** — that's Snowflake, not us. Pass
`lower=True` to `q()` if you'd rather have lower-case.

`datalake` also converts Snowflake's `Decimal` and `date` objects into real
`float64` and `datetime64` columns, so `.dt` accessors, formatting and
plotly all behave. If you need exact decimal arithmetic on money, pass
`convert=False` to keep the `Decimal` values.

⚠️ **The most recent month is partial** — it only covers days elapsed so
far, so it will always look like a collapse. The cells below drop it before
charting.

In [ ]:
# Drop the in-progress month so trends aren't misleading
latest = df["MONTH"].max()
full = df[df["MONTH"] < latest].copy()

print(f"excluded partial month: {latest:%Y-%m}")
print(f"{len(full)} rows remain, through {full['MONTH'].max():%Y-%m}")

## 3 — Headline numbers

Total collected per servicer over the period, with the share of the book.

In [ ]:
by_servicer = (
    full.groupby("SERVICER")
        .agg(
            collected=("COLLECTED", "sum"),
            transactions=("TRANSACTIONS", "sum"),
            months=("MONTH", "nunique"),
            median_payment=("MEDIAN_PAYMENT", "median"),
        )
        .sort_values("collected", ascending=False)
)
by_servicer["share_%"] = (
    100 * by_servicer["collected"] / by_servicer["collected"].sum()
)
by_servicer

## 4 — Trend over time

One line per servicer. A log y-axis keeps the smaller servicers readable
next to the largest — without it they flatten to zero.

In [ ]:
fig = px.line(
    full.sort_values("MONTH"),
    x="MONTH",
    y="COLLECTED",
    color="SERVICER",
    markers=True,
    log_y=True,
    title="Monthly collections by servicer",
)
fig.update_layout(yaxis_title="Collected (log scale)", xaxis_title=None)
fig

## 5 — Efficiency: collected per case

Total collected divided by distinct cases worked. Higher means each case is
yielding more — useful for comparing servicers handling different book
sizes, where absolute totals aren't comparable.

In [ ]:
fig = px.bar(
    full.sort_values("MONTH"),
    x="MONTH",
    y="COLLECTED_PER_CASE",
    color="SERVICER",
    barmode="group",
    title="Collected per case, by servicer",
)
fig.update_layout(yaxis_title="Collected per case", xaxis_title=None)
fig

## 6 — Month-on-month movement

`pivot` turns the long result into a month × servicer grid, which is the
natural shape for a report, and `pct_change` gives the trend.

In [ ]:
grid = (
    full.pivot(index="MONTH", columns="SERVICER", values="COLLECTED")
        .sort_index()
)
grid

In [ ]:
# Percent change vs the previous month.
# fill_method=None matters: some servicers are missing in some months,
# and the old default forward-filled those gaps, reporting a fake 0%
# change instead of a blank.
grid.pct_change(fill_method=None).mul(100)

## 7 — Save the results

Files land in this folder and are kept out of git by `.gitignore`. Uncomment
to write.

In [ ]:
# df.to_csv("2026-09-09-servicing-dashboard.csv", index=False)
# grid.to_excel("2026-09-09-servicing-by-month.xlsx")

## 8 — Writing your own queries

Short ones go inline. Bind real *values* as parameters (`%s`) rather than
formatting them in — safer, and Snowflake can reuse the query plan.

**This is where `{db}` placeholders belong.** Object names (database, schema,
table) cannot be bound as parameters, so `datalake` fills them with
`str.format`. Inline SQL like this is never opened by the Snowflake
extension, so the braces cost nothing here — whereas in a `.sql` file they
would make it unrunnable in the extension.

In [ ]:
datalake.q(
    """
    select
        rs.servicer_name  as servicer,
        count(*)          as transactions,
        sum(c.amount)     as collected
    from {db}.npl_common.fct_case_collection as c
    inner join {db}.npl_common.dim_reporting_system as rs
        on c.reporting_system_sk = rs.reporting_system_sk
    where c.collection_date between %s and %s
    group by 1
    order by collected desc
    """,
    ["2026-01-01", "2026-06-30"],   # bound parameters
    db=DB,
)

### Finding your way around the warehouse

Table names in Snowflake **drop** the `npl_` prefix the dbt model filenames
carry: the model `fct_npl_case_collection.sql` is
`PROD.NPL_COMMON.FCT_CASE_COLLECTION`. Never copy a name straight from the
repo — check it:

In [ ]:
datalake.q(f"show tables in schema {DB}.npl_common")[["name", "rows"]]

In [ ]:
datalake.q(f"describe table {DB}.npl_common.fct_case_collection")[["name", "type"]]

`describe table` gives you names and types, but **not the documentation**. `datalake.describe_column()` does — and it searches
*across* tables, so it answers the question you actually have: **which
table is this column in, what type is it there, and what do the docs
say about both?**

It is [../work/general_queries/data_dictionary.sql](../work/general_queries/data_dictionary.sql)
narrowed to one column. The descriptions are the dbt model docs from
`brocc-data`, which dbt pushes into Snowflake as native `COMMENT`s on
every build — so Snowflake itself is the dictionary.


In [ ]:
# The join key this dashboard's query relies on. One row per table
# that carries it -- which is how you see it is not the same everywhere.
datalake.describe_column("reporting_system_sk")


⚠️ **Read that output before trusting a join.** `REPORTING_SYSTEM_SK`
appears in several tables, and it is **100% NULL in
`FCT_PORTFOLIO_SNAPSHOT_MONTHLY`** — a known data-quality problem in the
warehouse, not something this notebook introduces. Joining that table on
this key silently returns nothing. This dashboard joins
`FCT_CASE_COLLECTION` instead, where the key is populated.

That is exactly the class of problem this function is for: the column
exists, the types match, and the join still produces garbage.


Two related helpers, for when you *don't* have the exact column name:

| Function | Use when |
|---|---|
| `describe_column("gbv_amount")` | you know the name, want to know where it lives |
| `describe_column("gbv", exact=False)` | you know part of the name |
| `find_columns("servicer")` | you know the *concept* — searches descriptions too |
| `dictionary("fct_case_collection")` | you know the table, want all its columns |


In [ ]:
# Partial-name search: every column whose name contains "collect"
datalake.describe_column("collect", exact=False)[
    ["TABLE_NAME", "COLUMN_NAME", "DATA_TYPE", "COLUMN_DESCRIPTION"]
]


---

## Does any of this write to Snowflake?

**No.** Three independent reasons:

1. **Every query here is a `SELECT`.** The only other statement `datalake`
   ever issues is `use warehouse`, which sets a session default and changes
   no data.
2. **Your role can't write.** `NPL_ANALYST` holds only `SELECT` and `USAGE`
   privileges, inherited from three read-only roles. It has no `INSERT`,
   `UPDATE`, `DELETE`, `CREATE` or `MODIFY` anywhere — an attempted write
   fails on privileges.
3. **`to_csv` / `to_excel` write to your laptop**, not the warehouse.

`datalake.execute()` exists for statements that return nothing, but with a
read-only role there is nothing destructive it can do. If you ever *do* need
to write, that's a permissions request to the data team — and it should
happen through dbt in `brocc-data`, not from a notebook.

### Cheat sheet

| Call | What it does |
|------|--------------|
| `datalake.q(sql)` | Run a query → pandas DataFrame |
| `datalake.q("example_servicing-dashboard.sql", db=DB, months=12)` | Run the file next to this notebook |
| `datalake.q("name", db=DB)` | Run `../work/name.sql` |
| `datalake.q(sql, lower=True)` | Lower-case the column names |
| `datalake.q(sql, params)` | Run with bound parameters |
| `datalake.read_sql("name", db=DB)` | Read a `.sql` file without running it |
| `datalake.session_info()` | Account, user, role, warehouse |
| `datalake.warehouses()` | Warehouses your role can use |
| `datalake.q(sql, convert=False)` | Keep exact `Decimal` values instead of float |
| `datalake.reset()` | Reconnect after your login token expires |

Full docs and troubleshooting: [../README.md](../README.md)